# EDA_AGENT: structured Cifrium LMS EDA and feature pipeline

Goal: build a professional, auditable, ML-oriented EDA in grain `(user_id, course_id)`.

Main principles:
- keep source notebooks and raw data read-only;
- reuse useful logic from `EDA.ipynb`, `Resolve_id.ipynb`, `Resolve_AGENT.ipynb`;
- split rough preprocessing table by table;
- restore `(user_id, course_id) -> users_course_id` in a dedicated block close to `Resolve_AGENT.ipynb`;
- validate every important join;
- build interpretable feature blocks aligned with churn/success analysis;
- save only ASCII outputs to `data/AGENTS`.


## 0. Safety and notebook map

Protected files are read-only references: `notebooks/EDA.ipynb`, `notebooks/Resolve_id.ipynb`, `notebooks/Resolve_AGENT.ipynb`, `data/raw`, `README.md`, `Hipotises.md`.

This notebook writes outputs only to `data/AGENTS`.

Notebook map:
1. Environment and config.
2. Data loading and source audit.
3. Rough preprocessing by table.
4. Key restoration block.
5. Resolution and join audit.
6. Table-by-table feature engineering.
7. Hypothesis checks with metrics and plots.
8. Final merge and final audit.
9. Export artifacts and final summary.


## 1. Environment, imports, config

`UNRESOLVED_MODE` controls unresolved key handling.

`keep`: keep unresolved `(user_id, course_id)` rows. For downstream course-activity features, unresolved rows keep `NA` because the bridge is unknown; resolved rows with no events get zero-like event counts.

`drop`: drop unresolved rows before final modeling table export.


In [ ]:
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 260)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "AGENTS"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

UNRESOLVED_MODE = "keep"
TARGET_INACTIVITY_DAYS = 30
MIN_COURSE_ROWS_FOR_COURSE_AUDIT = 10
USER_ANSWERS_CHUNK_SIZE = 500_000

if UNRESOLVED_MODE not in {"keep", "drop"}:
    raise ValueError("UNRESOLVED_MODE must be keep or drop")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("UNRESOLVED_MODE:", UNRESOLVED_MODE)


In [ ]:
def to_int_id(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        numeric = pd.to_numeric(series, errors="coerce")
    else:
        numeric = pd.to_numeric(series.astype("string").str.replace(",", "", regex=False), errors="coerce")
    return numeric.astype("Int64")

def to_bool01(series: pd.Series) -> pd.Series:
    return series.astype("string").str.strip().str.lower().isin(["true", "1", "yes"]).astype("Int8")

def safe_div(num, den):
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    return np.where(den > 0, num / den, np.nan)

def clean_name(value) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unknown"

def missingness_table(df: pd.DataFrame, top_n: int = 200) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_share": df.isna().mean().values,
        "dtype": [str(df[c].dtype) for c in df.columns],
    }).sort_values(["missing_share", "missing_count"], ascending=False).head(top_n).reset_index(drop=True)

def merge_many_to_one(base: pd.DataFrame, add: pd.DataFrame, keys, name: str, audit_rows: list, how: str = "left") -> pd.DataFrame:
    before_rows = len(base)
    out = base.merge(add, on=keys, how=how, validate="many_to_one")
    audit_rows.append({
        "merge_name": name,
        "keys": "|".join(keys),
        "how": how,
        "rows_before": before_rows,
        "rows_after": len(out),
        "row_delta": len(out) - before_rows,
        "base_duplicate_keys_before": int(base.duplicated(keys).sum()),
        "add_duplicate_keys": int(add.duplicated(keys).sum()),
        "duplicate_keys_after": int(out.duplicated(keys).sum()),
    })
    return out

def plot_bar(series, title, xlabel, ylabel, figsize=(6, 3)):
    plt.figure(figsize=figsize)
    series.plot(kind="bar")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()

def display_head(df, n=10):
    display(df.head(n))


## 2. Data loading setup

This block mirrors the style of the source EDA: file map, date column map, and one loader function.

`user_answers` is intentionally not loaded here because it is very large; it is processed later in chunks as user-level enrichment.


In [ ]:
FILES = {
    "users_courses": "users_courses.csv",
    "users": "users.csv",
    "lessons": "lessons.csv",
    "user_lessons": "user_lessons.csv",
    "wk_users_courses_actions": "wk_users_courses_actions.csv",
    "user_access_histories": "user_access_histories.csv",
    "user_activity_histories": "user_activity_histories.csv",
    "user_trainings": "user_trainings.csv",
    "wk_media_view_sessions": "wk_media_view_sessions.csv",
    "user_award_badges": "user_award_badges.csv",
    "award_badges": "award_badges.csv",
}

USECOLS = {
    "users_courses": ["user_id", "course_id", "state", "created_at", "updated_at", "access_finished_at", "wk_points", "wk_max_points", "wk_max_viewable_lessons", "wk_max_task_count", "wk_officially_started_at", "wk_course_completed_at"],
    "users": ["id", "type", "created_at", "sign_in_count", "grade_id", "subscribed", "timezone", "d_wk_school_id", "d_wk_municipal_id", "d_wk_region_id", "wk_gender"],
    "lessons": ["course_id", "conspect_expected", "task_expected", "lesson_number", "wk_max_points", "wk_task_count", "wk_survival_training_expected", "wk_scratch_playground_enabled", "wk_attendance_tracking_enabled", "wk_video_duration"],
    "user_lessons": ["Unnamed: 0", "user_id", "lesson_id", "group_id", "video_visited", "translation_visited", "users_course_id", "solved", "solved_tasks_count", "wk_points", "video_viewed", "wk_solved_task_count"],
    "wk_users_courses_actions": ["user_id", "users_course_id", "action", "created_at", "updated_at", "lesson_id"],
    "user_access_histories": ["users_course_id", "access_started_at", "access_expired_at", "activator_class"],
    "user_activity_histories": ["user_lesson_id", "action", "created_at"],
    "user_trainings": ["user_id", "training_id", "solved_tasks_count", "earned_points", "type", "state", "submitted_answers_count", "started_at", "finished_at", "attempts", "mark", "mark_saved_at"],
    "wk_media_view_sessions": ["resource_type", "viewer_id", "segments_total", "viewed_segments_count", "started_at", "kind"],
    "user_award_badges": ["award_badge_id", "user_id", "created_at"],
    "award_badges": ["Unnamed: 0", "name", "title", "level", "quota", "special"],
}

DATE_COLS = {
    "users_courses": ["created_at", "updated_at", "access_finished_at", "wk_officially_started_at", "wk_course_completed_at"],
    "users": ["created_at"],
    "wk_users_courses_actions": ["created_at", "updated_at"],
    "user_access_histories": ["access_started_at", "access_expired_at"],
    "user_activity_histories": ["created_at"],
    "user_trainings": ["started_at", "finished_at", "mark_saved_at"],
    "wk_media_view_sessions": ["started_at"],
    "user_award_badges": ["created_at"],
}

def load_tables(data_dir: Path, files: dict, usecols: dict) -> tuple[dict, pd.DataFrame]:
    dfs_local = {}
    overview = []
    for table_name, filename in files.items():
        df = pd.read_csv(data_dir / filename, usecols=usecols[table_name], encoding="utf-8", low_memory=False)
        dfs_local[table_name] = df
        overview.append({"table": table_name, "rows": len(df), "cols": df.shape[1], "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)})
    return dfs_local, pd.DataFrame(overview).sort_values("table").reset_index(drop=True)

dfs, source_overview_AGENTS = load_tables(DATA_DIR, FILES, USECOLS)
display(source_overview_AGENTS)


## 3. Source table overview and duplicate audit

Before preprocessing, check source shapes and full-row duplicates. Event tables can have repeated-looking rows, so this block reports duplicates but does not blindly drop them.


In [ ]:
schema_rows = []
for csv_path in sorted(DATA_DIR.glob("*.csv")):
    cols = pd.read_csv(csv_path, nrows=0, encoding="utf-8").columns.tolist()
    schema_rows.append({"table": csv_path.stem, "column_count": len(cols), "columns_preview": "|".join([clean_name(c) for c in cols[:8]])})
source_schema_AGENTS = pd.DataFrame(schema_rows)
display(source_schema_AGENTS)

duplicate_audit_AGENTS = pd.DataFrame([
    {"table": name, "rows": len(df), "full_duplicate_rows": int(df.duplicated().sum()), "full_duplicate_share": float(df.duplicated().mean()) if len(df) else 0.0}
    for name, df in dfs.items()
]).sort_values("full_duplicate_rows", ascending=False).reset_index(drop=True)
display(duplicate_audit_AGENTS)


## 4. Rough preprocessing by table

This section follows the source EDA style: each table is processed in its own cell. The goal is not to over-refactor, but to keep all cleaning decisions visible and auditable.


### 4.1 `users_courses` rough preprocessing

`users_courses` is the master table. It defines the target grain `(user_id, course_id)` and keeps course status/progress fields. `state` is not used as final target; it is kept only as a status/audit column.


In [ ]:
uc = dfs["users_courses"].copy()
for col in ["user_id", "course_id"]:
    uc[col] = to_int_id(uc[col])
for col in ["wk_points", "wk_max_points", "wk_max_viewable_lessons", "wk_max_task_count"]:
    uc[col] = pd.to_numeric(uc[col], errors="coerce")
for col in DATE_COLS["users_courses"]:
    uc[col] = pd.to_datetime(uc[col], errors="coerce")
uc["uc_is_active_state"] = uc["state"].astype("string").str.lower().eq("active").astype("Int8")
uc["uc_is_completed_flag"] = uc["wk_course_completed_at"].notna().astype("Int8")
uc["uc_points_ratio"] = safe_div(uc["wk_points"], uc["wk_max_points"])
uc["uc_access_window_days"] = (uc["access_finished_at"] - uc["created_at"]).dt.days
uc["uc_days_start_to_official_start"] = (uc["wk_officially_started_at"] - uc["created_at"]).dt.days
uc["uc_task_capacity_ratio"] = safe_div(uc["wk_max_task_count"], uc["wk_max_viewable_lessons"])
dfs["users_courses"] = uc
users_courses_preprocess_audit = pd.DataFrame([{"metric": "rows", "value": len(uc)}, {"metric": "unique_user_course_pairs", "value": uc[["user_id", "course_id"]].drop_duplicates().shape[0]}, {"metric": "duplicate_user_course_pairs", "value": int(uc.duplicated(["user_id", "course_id"]).sum())}, {"metric": "state_active_share", "value": float(uc["uc_is_active_state"].mean())}, {"metric": "course_completed_share", "value": float(uc["uc_is_completed_flag"].mean())}])
display(users_courses_preprocess_audit)


### 4.2 `users` rough preprocessing and teacher ids

Teacher/account-agent filtering follows the source EDA: `users.type` containing `Agent` is treated as teacher-like account.


In [ ]:
users = dfs["users"].copy()
for col in ["id", "sign_in_count", "grade_id", "d_wk_school_id", "d_wk_municipal_id", "d_wk_region_id"]:
    users[col] = to_int_id(users[col])
users["created_at"] = pd.to_datetime(users["created_at"], errors="coerce")
users["user_id"] = users["id"].copy()
users["is_teacher_agent"] = users["type"].astype("string").str.contains("Agent", na=False)
teacher_ids = set(users.loc[users["is_teacher_agent"], "user_id"].dropna().astype("int64"))
users["user_subscribed_flag"] = to_bool01(users["subscribed"])
dfs["users"] = users
users_preprocess_audit = pd.DataFrame([{"metric": "users_rows", "value": len(users)}, {"metric": "teacher_agent_users", "value": len(teacher_ids)}, {"metric": "pupil_like_users", "value": int((~users["is_teacher_agent"]).sum())}])
display(users_preprocess_audit)


### 4.3 `lessons` rough preprocessing

`lessons` is a course-level reference source. It is useful for compact course metadata, but it should not overwrite `users_courses` fields without audit because source EDA found mismatches.


In [ ]:
lessons = dfs["lessons"].copy()
lessons["course_id"] = to_int_id(lessons["course_id"])
for col in ["wk_max_points", "wk_task_count", "wk_video_duration", "lesson_number"]:
    lessons[col] = pd.to_numeric(lessons[col], errors="coerce")
for col in ["conspect_expected", "task_expected", "wk_survival_training_expected", "wk_scratch_playground_enabled", "wk_attendance_tracking_enabled"]:
    lessons[col] = to_bool01(lessons[col])
dfs["lessons"] = lessons
lessons_preprocess_audit = pd.DataFrame([{"metric": "lessons_rows", "value": len(lessons)}, {"metric": "course_id_nunique", "value": lessons["course_id"].nunique()}, {"metric": "task_expected_share", "value": float(lessons["task_expected"].mean())}])
display(lessons_preprocess_audit)


### 4.4 `user_lessons` rough preprocessing

`user_lessons` is the main lesson-progress table and one of the strongest sources for restoring `users_course_id`. The technical CSV row id is kept as `user_lesson_row_id` for the `user_activity_histories` bridge audit.


In [ ]:
ul = dfs["user_lessons"].copy().rename(columns={"Unnamed: 0": "user_lesson_row_id"})
for col in ["user_lesson_row_id", "user_id", "lesson_id", "group_id", "users_course_id"]:
    ul[col] = to_int_id(ul[col])
for col in ["video_visited", "translation_visited", "solved", "video_viewed"]:
    ul[col] = to_bool01(ul[col])
for col in ["solved_tasks_count", "wk_points", "wk_solved_task_count"]:
    ul[col] = pd.to_numeric(ul[col], errors="coerce")
dfs["user_lessons"] = ul
user_lessons_preprocess_audit = pd.DataFrame([{"metric": "user_lessons_rows", "value": len(ul)}, {"metric": "users_course_id_nunique", "value": ul["users_course_id"].nunique()}, {"metric": "max_user_id_per_users_course_id", "value": ul.groupby("users_course_id")["user_id"].nunique().max()}, {"metric": "video_visited_share", "value": float(ul["video_visited"].mean())}])
display(user_lessons_preprocess_audit)


### 4.5 `wk_users_courses_actions` rough preprocessing

This is the main course-action event log. It supports recency, frequency, diversity, activity gaps and target-related inactivity logic.


In [ ]:
wka = dfs["wk_users_courses_actions"].copy()
for col in ["user_id", "users_course_id", "lesson_id"]:
    wka[col] = to_int_id(wka[col])
for col in ["created_at", "updated_at"]:
    wka[col] = pd.to_datetime(wka[col], errors="coerce")
wka["action_clean"] = wka["action"].astype("string").map(clean_name)
dfs["wk_users_courses_actions"] = wka
wka_preprocess_audit = pd.DataFrame([{"metric": "wka_rows", "value": len(wka)}, {"metric": "users_course_id_nunique", "value": wka["users_course_id"].nunique()}, {"metric": "max_user_id_per_users_course_id", "value": wka.dropna(subset=["users_course_id"]).groupby("users_course_id")["user_id"].nunique().max()}, {"metric": "action_nunique", "value": wka["action_clean"].nunique()}])
display(wka_preprocess_audit)
display(wka["action_clean"].value_counts().rename_axis("action_clean").reset_index(name="rows"))


### 4.6 `user_access_histories` rough preprocessing

This table has `users_course_id` but no `user_id`. It is useful after key restoration and for access-duration hypotheses.


In [ ]:
access = dfs["user_access_histories"].copy()
access["users_course_id"] = to_int_id(access["users_course_id"])
for col in ["access_started_at", "access_expired_at"]:
    access[col] = pd.to_datetime(access[col], errors="coerce")
access["activator_clean"] = access["activator_class"].astype("string").str.split("::").str[-1].map(clean_name)
access["access_interval_days"] = (access["access_expired_at"] - access["access_started_at"]).dt.days
dfs["user_access_histories"] = access
access_preprocess_audit = pd.DataFrame([{"metric": "access_rows", "value": len(access)}, {"metric": "users_course_id_nunique", "value": access["users_course_id"].nunique()}, {"metric": "activator_nunique", "value": access["activator_clean"].nunique()}, {"metric": "interval_days_mean", "value": float(access["access_interval_days"].mean())}])
display(access_preprocess_audit)
display(access["activator_clean"].value_counts().rename_axis("activator_clean").reset_index(name="rows"))


### 4.7 `user_activity_histories` rough preprocessing

Source EDA treated this table cautiously. Here it is not used as the main activity log. It is tested as auxiliary data through `user_lesson_id -> user_lessons.user_lesson_row_id`.


In [ ]:
uahist = dfs["user_activity_histories"].copy()
uahist["user_lesson_id"] = to_int_id(uahist["user_lesson_id"])
uahist["created_at"] = pd.to_datetime(uahist["created_at"], errors="coerce")
uahist["action_clean"] = uahist["action"].astype("string").map(clean_name)
dfs["user_activity_histories"] = uahist
uahist_preprocess_audit = pd.DataFrame([{"metric": "uahist_rows", "value": len(uahist)}, {"metric": "user_lesson_id_nunique", "value": uahist["user_lesson_id"].nunique()}, {"metric": "action_nunique", "value": uahist["action_clean"].nunique()}])
display(uahist_preprocess_audit)
display(uahist["action_clean"].value_counts().rename_axis("action_clean").reset_index(name="rows"))


### 4.8 `user_trainings` rough preprocessing

`user_trainings` has no validated course bridge in raw columns, so it is used as user-level enrichment. Marks are useful for success-related proxy checks.


In [ ]:
train = dfs["user_trainings"].copy()
for col in ["user_id", "training_id"]:
    train[col] = to_int_id(train[col])
for col in ["started_at", "finished_at", "mark_saved_at"]:
    train[col] = pd.to_datetime(train[col], errors="coerce")
for col in ["solved_tasks_count", "earned_points", "submitted_answers_count", "attempts", "mark"]:
    train[col] = pd.to_numeric(train[col], errors="coerce")
train["type_clean"] = train["type"].astype("string").str.split("::").str[-1].map(clean_name)
train["state_clean"] = train["state"].astype("string").map(clean_name)
train["training_duration_min"] = (train["finished_at"] - train["started_at"]).dt.total_seconds() / 60
train["mark_ge4_flag"] = (train["mark"] >= 4).astype("Int8")
dfs["user_trainings"] = train
training_preprocess_audit = pd.DataFrame([{"metric": "training_rows", "value": len(train)}, {"metric": "user_id_nunique", "value": train["user_id"].nunique()}, {"metric": "mark_nonnull_share", "value": float(train["mark"].notna().mean())}, {"metric": "mark_ge4_share_on_nonnull", "value": float(train.loc[train["mark"].notna(), "mark_ge4_flag"].mean())}])
display(training_preprocess_audit)
display(train["type_clean"].value_counts().rename_axis("type_clean").reset_index(name="rows"))


### 4.9 `wk_media_view_sessions` rough preprocessing

Media sessions are joined by `viewer_id -> user_id`, so they are user-level engagement features, not course-specific features.


In [ ]:
media = dfs["wk_media_view_sessions"].copy().rename(columns={"viewer_id": "user_id"})
media["user_id"] = to_int_id(media["user_id"])
media["started_at"] = pd.to_datetime(media["started_at"], errors="coerce")
for col in ["segments_total", "viewed_segments_count"]:
    media[col] = pd.to_numeric(media[col], errors="coerce")
media["kind_clean"] = media["kind"].astype("string").map(clean_name)
media["media_view_ratio_row"] = safe_div(media["viewed_segments_count"], media["segments_total"])
dfs["wk_media_view_sessions"] = media
media_preprocess_audit = pd.DataFrame([{"metric": "media_rows", "value": len(media)}, {"metric": "user_id_nunique", "value": media["user_id"].nunique()}, {"metric": "kind_nunique", "value": media["kind_clean"].nunique()}, {"metric": "view_ratio_mean", "value": float(np.nanmean(media["media_view_ratio_row"]))}])
display(media_preprocess_audit)
display(media["kind_clean"].value_counts().rename_axis("kind_clean").reset_index(name="rows"))


### 4.10 `user_award_badges` and `award_badges` rough preprocessing

Awards are user-level motivation/engagement proxies. Raw image/title fields are not used as features.


In [ ]:
award_ref = dfs["award_badges"].copy().rename(columns={"Unnamed: 0": "award_badge_id"})
award_ref["award_badge_id"] = to_int_id(award_ref["award_badge_id"])
for col in ["level", "quota"]:
    award_ref[col] = pd.to_numeric(award_ref[col], errors="coerce")
award_ref["special"] = to_bool01(award_ref["special"])
award_ref["award_name_clean"] = award_ref["name"].astype("string").str.split("::").str[-1].map(clean_name)

award_events = dfs["user_award_badges"].copy()
award_events["award_badge_id"] = to_int_id(award_events["award_badge_id"])
award_events["user_id"] = to_int_id(award_events["user_id"])
award_events["created_at"] = pd.to_datetime(award_events["created_at"], errors="coerce")
dfs["award_badges"] = award_ref
dfs["user_award_badges"] = award_events
award_preprocess_audit = pd.DataFrame([{"metric": "award_ref_rows", "value": len(award_ref)}, {"metric": "award_events_rows", "value": len(award_events)}, {"metric": "award_user_nunique", "value": award_events["user_id"].nunique()}])
display(award_preprocess_audit)
display(award_ref[["award_badge_id", "award_name_clean", "level", "quota", "special"]])


### 4.11 Teacher filter applied to user-keyed tables

This step follows source EDA and removes teacher/account-agent users from all tables that have a user-like key. `user_access_histories` is not filtered here because it has no `user_id`.


In [ ]:
teacher_filter_audit_rows = []
for table in ["users_courses", "user_lessons", "wk_users_courses_actions", "user_trainings", "user_award_badges"]:
    before = len(dfs[table])
    dfs[table] = dfs[table].loc[~dfs[table]["user_id"].isin(teacher_ids)].copy()
    teacher_filter_audit_rows.append({"table": table, "checked_column": "user_id", "rows_before": before, "rows_after": len(dfs[table]), "removed_rows": before - len(dfs[table])})
before = len(dfs["wk_media_view_sessions"])
dfs["wk_media_view_sessions"] = dfs["wk_media_view_sessions"].loc[~dfs["wk_media_view_sessions"]["user_id"].isin(teacher_ids)].copy()
teacher_filter_audit_rows.append({"table": "wk_media_view_sessions", "checked_column": "user_id", "rows_before": before, "rows_after": len(dfs["wk_media_view_sessions"]), "removed_rows": before - len(dfs["wk_media_view_sessions"])})
before = len(dfs["users"])
dfs["users"] = dfs["users"].loc[~dfs["users"]["user_id"].isin(teacher_ids)].copy()
teacher_filter_audit_rows.append({"table": "users", "checked_column": "user_id", "rows_before": before, "rows_after": len(dfs["users"]), "removed_rows": before - len(dfs["users"])})
teacher_filter_audit_AGENTS = pd.DataFrame(teacher_filter_audit_rows).sort_values("removed_rows", ascending=False).reset_index(drop=True)
display(teacher_filter_audit_AGENTS)


## 5. Key restoration block: `(user_id, course_id) -> users_course_id`

This block intentionally follows `Resolve_AGENT.ipynb` closely. It uses the same evidence families:
- `points` signal;
- `lesson_count` signal;
- `access_date` signal;
- `group_intersection` signal based on `(lesson_id, group_id)`;
- `user_singleton` signal.

Only globally non-conflicting mappings are accepted.


### 5.1 Base objects for key restoration

The base table is all unique student `(user_id, course_id)` pairs from `users_courses` after teacher filtering. Activity ids are taken from `user_lessons` and `wk_users_courses_actions`.


In [ ]:
uc_pairs = dfs["users_courses"][["user_id", "course_id", "wk_points", "wk_max_viewable_lessons", "access_finished_at"]].dropna(subset=["user_id", "course_id"]).drop_duplicates(subset=["user_id", "course_id"]).reset_index(drop=True)
ul_base = dfs["user_lessons"][["user_id", "lesson_id", "group_id", "users_course_id", "wk_points"]].dropna(subset=["user_id", "users_course_id"]).drop_duplicates().reset_index(drop=True)
wka_pairs = dfs["wk_users_courses_actions"][["user_id", "users_course_id"]].dropna(subset=["user_id", "users_course_id"]).drop_duplicates().reset_index(drop=True)
access_base = dfs["user_access_histories"][["users_course_id", "access_expired_at"]].dropna(subset=["users_course_id"]).drop_duplicates()
all_user_ucid = pd.concat([ul_base[["user_id", "users_course_id"]], wka_pairs], ignore_index=True).drop_duplicates().reset_index(drop=True)
key_base_audit_AGENTS = pd.DataFrame([{"metric": "uc_pairs", "value": len(uc_pairs)}, {"metric": "ul_user_ucid_pairs", "value": ul_base[["user_id", "users_course_id"]].drop_duplicates().shape[0]}, {"metric": "wka_user_ucid_pairs", "value": len(wka_pairs)}, {"metric": "combined_user_ucid_pairs", "value": len(all_user_ucid)}, {"metric": "max_users_per_ucid_combined", "value": all_user_ucid.groupby("users_course_id")["user_id"].nunique().max()}])
display(key_base_audit_AGENTS)


### 5.2 Signals: points, lesson_count, access_date

These signals reproduce the compact candidate logic from `Resolve_AGENT.ipynb`: build all user-local candidate edges and keep matches supported by progress/date evidence.


In [ ]:
ul_agg_for_key = ul_base.groupby(["user_id", "users_course_id"], dropna=False).agg(ul_rows=("lesson_id", "size"), ul_lesson_nunique=("lesson_id", "nunique"), ul_points_sum=("wk_points", "sum")).reset_index()
access_agg_for_key = access_base.assign(access_expired_date=lambda x: x["access_expired_at"].dt.normalize()).groupby("users_course_id").agg(access_expired_min=("access_expired_date", "min"), access_expired_max=("access_expired_date", "max")).reset_index()
candidate_cross = all_user_ucid.merge(uc_pairs.assign(access_finished_date=lambda x: x["access_finished_at"].dt.normalize()), on="user_id", how="inner").merge(ul_agg_for_key, on=["user_id", "users_course_id"], how="left").merge(access_agg_for_key, on="users_course_id", how="left")
candidate_cross["points_match"] = ((candidate_cross["wk_points"].fillna(0) - candidate_cross["ul_points_sum"].fillna(0)).abs().le(1e-6) & candidate_cross["ul_rows"].notna())
candidate_cross["lesson_count_match"] = candidate_cross["wk_max_viewable_lessons"].notna() & candidate_cross["ul_lesson_nunique"].notna() & candidate_cross["wk_max_viewable_lessons"].eq(candidate_cross["ul_lesson_nunique"])
candidate_cross["access_date_match"] = candidate_cross["access_finished_date"].notna() & (candidate_cross["access_finished_date"].eq(candidate_cross["access_expired_min"]) | candidate_cross["access_finished_date"].eq(candidate_cross["access_expired_max"]))
signal_candidate_summary_AGENTS = []
for signal_name in ["points_match", "lesson_count_match", "access_date_match"]:
    edges = candidate_cross.loc[candidate_cross[signal_name], ["user_id", "course_id", "users_course_id"]].drop_duplicates()
    signal_candidate_summary_AGENTS.append({"signal": signal_name, "candidate_edges": len(edges), "candidate_user_ucid": edges[["user_id", "users_course_id"]].drop_duplicates().shape[0], "candidate_user_course": edges[["user_id", "course_id"]].drop_duplicates().shape[0]})
signal_candidate_summary_AGENTS = pd.DataFrame(signal_candidate_summary_AGENTS)
display(signal_candidate_summary_AGENTS)


### 5.3 Signal: group_intersection

This is the group-based logic from `Resolve_id.ipynb`: for each `(lesson_id, group_id)`, find courses shared by all group members, then require support across all lesson/group rows of a `(user_id, users_course_id)` pair.


In [ ]:
ul_group_base = ul_base[["user_id", "lesson_id", "group_id", "users_course_id"]].dropna(subset=["lesson_id", "group_id"]).drop_duplicates()
group_members = ul_group_base[["lesson_id", "group_id", "user_id"]].drop_duplicates()
group_sizes = group_members.groupby(["lesson_id", "group_id"])["user_id"].nunique().rename("group_size").reset_index()
group_member_courses = group_members.merge(uc_pairs[["user_id", "course_id"]], on="user_id", how="left").dropna(subset=["course_id"])
group_course_support = group_member_courses.groupby(["lesson_id", "group_id", "course_id"])["user_id"].nunique().rename("course_support").reset_index().merge(group_sizes, on=["lesson_id", "group_id"], how="left")
group_course_intersection = group_course_support.loc[group_course_support["course_support"].eq(group_course_support["group_size"]), ["lesson_id", "group_id", "course_id"]].copy()
ul_group_candidates = ul_group_base.merge(group_course_intersection, on=["lesson_id", "group_id"], how="left").dropna(subset=["course_id"])
user_ucid_group_count = ul_group_base.groupby(["user_id", "users_course_id"]).size().rename("pair_group_count").reset_index()
group_edges = ul_group_candidates.drop_duplicates(["user_id", "users_course_id", "lesson_id", "group_id", "course_id"]).groupby(["user_id", "users_course_id", "course_id"]).size().rename("support").reset_index().merge(user_ucid_group_count, on=["user_id", "users_course_id"], how="left").query("support == pair_group_count")[["user_id", "course_id", "users_course_id"]].drop_duplicates()
group_signal_audit_AGENTS = pd.DataFrame([{"metric": "lesson_group_count", "value": group_sizes.shape[0]}, {"metric": "lesson_groups_with_intersection", "value": group_course_intersection[["lesson_id", "group_id"]].drop_duplicates().shape[0]}, {"metric": "group_candidate_edges", "value": len(group_edges)}, {"metric": "group_candidate_user_ucid", "value": group_edges[["user_id", "users_course_id"]].drop_duplicates().shape[0]}])
display(group_signal_audit_AGENTS)
plot_bar(group_course_intersection.groupby(["lesson_id", "group_id"])["course_id"].nunique().value_counts().sort_index().head(20), "Group-intersection candidate count", "candidate course count", "lesson-group count")


### 5.4 Global uniqueness and conflict removal

A mapping is accepted only if it is one-to-one inside a method and remains conflict-free after method union.


In [ ]:
def force_unique(edges: pd.DataFrame) -> pd.DataFrame:
    keys = ["user_id", "course_id", "users_course_id"]
    edges = edges[keys].drop_duplicates().copy()
    if edges.empty:
        return edges
    c1 = edges.groupby(["user_id", "users_course_id"])["course_id"].nunique().rename("course_candidates").reset_index()
    c2 = edges.groupby(["user_id", "course_id"])["users_course_id"].nunique().rename("ucid_candidates").reset_index()
    tmp = edges.merge(c1, on=["user_id", "users_course_id"]).merge(c2, on=["user_id", "course_id"])
    return tmp.loc[tmp["course_candidates"].eq(1) & tmp["ucid_candidates"].eq(1), keys].drop_duplicates()

evidence_parts = []
for label, col in [("points", "points_match"), ("lesson_count", "lesson_count_match"), ("access_date", "access_date_match")]:
    part = candidate_cross.loc[candidate_cross[col], ["user_id", "course_id", "users_course_id"]].drop_duplicates().copy()
    part["evidence"] = label
    evidence_parts.append(part)
group_part = group_edges.copy(); group_part["evidence"] = "group_intersection"; evidence_parts.append(group_part)
evidence_long = pd.concat(evidence_parts, ignore_index=True).drop_duplicates()
edge_evidence = evidence_long.groupby(["user_id", "course_id", "users_course_id"])["evidence"].agg(lambda s: "|".join(sorted(set(s)))).reset_index()
edge_evidence["evidence_count"] = edge_evidence["evidence"].str.count(r"\|") + 1
method_edges = {"points_unique": force_unique(edge_evidence.loc[edge_evidence["evidence"].str.contains("points", regex=False)]), "date_unique": force_unique(edge_evidence.loc[edge_evidence["evidence"].str.contains("access_date", regex=False)]), "group_unique": force_unique(edge_evidence.loc[edge_evidence["evidence"].str.contains("group_intersection", regex=False)]), "evidence2_unique": force_unique(edge_evidence.loc[edge_evidence["evidence_count"].ge(2)])}
user_course_count = uc_pairs.groupby("user_id")["course_id"].nunique().rename("course_count").reset_index()
user_ucid_count = all_user_ucid.groupby("user_id")["users_course_id"].nunique().rename("observed_users_course_id_count").reset_index()
method_edges["user_singleton"] = user_course_count.merge(user_ucid_count, on="user_id", how="inner").query("course_count == 1 and observed_users_course_id_count == 1")[["user_id"]].merge(uc_pairs[["user_id", "course_id"]], on="user_id").merge(all_user_ucid, on="user_id")[["user_id", "course_id", "users_course_id"]].drop_duplicates()
method_union = pd.concat([df.assign(resolution_method=name) for name, df in method_edges.items()], ignore_index=True).drop_duplicates()
edge_methods = method_union.groupby(["user_id", "course_id", "users_course_id"])["resolution_method"].agg(lambda s: "|".join(sorted(set(s)))).reset_index()
conflict_user_course = edge_methods.groupby(["user_id", "course_id"])["users_course_id"].nunique().rename("users_course_id_count").reset_index().query("users_course_id_count > 1")
conflict_user_ucid = edge_methods.groupby(["user_id", "users_course_id"])["course_id"].nunique().rename("course_id_count").reset_index().query("course_id_count > 1")
clean_mapping = edge_methods.merge(conflict_user_course[["user_id", "course_id"]].assign(_conflict_uc=1), on=["user_id", "course_id"], how="left").merge(conflict_user_ucid[["user_id", "users_course_id"]].assign(_conflict_id=1), on=["user_id", "users_course_id"], how="left")
clean_mapping = clean_mapping.loc[clean_mapping["_conflict_uc"].isna() & clean_mapping["_conflict_id"].isna()].drop(columns=["_conflict_uc", "_conflict_id"])
method_acceptance_audit_AGENTS = pd.DataFrame([{"method": k, "accepted_edges_before_global_conflict_check": len(v)} for k, v in method_edges.items()]).sort_values("accepted_edges_before_global_conflict_check", ascending=False)
display(method_acceptance_audit_AGENTS)
display(conflict_user_course)
display(conflict_user_ucid)


### 5.5 Final mapping and resolution audit

The final mapping is a left join to all base `(user_id, course_id)` pairs. Unresolved rows are kept or dropped later based on `UNRESOLVED_MODE`.


In [ ]:
resolution_mapping_AGENTS = uc_pairs[["user_id", "course_id"]].merge(clean_mapping, on=["user_id", "course_id"], how="left").sort_values(["user_id", "course_id"]).reset_index(drop=True)
resolution_mapping_AGENTS["users_course_id"] = pd.to_numeric(resolution_mapping_AGENTS["users_course_id"], errors="coerce").astype("Int64")
resolution_mapping_AGENTS["users_course_id_resolved_flag"] = resolution_mapping_AGENTS["users_course_id"].notna().astype("Int8")
resolution_mapping_AGENTS["unresolved_handling_mode"] = UNRESOLVED_MODE
resolution_mapping_AGENTS["maybe_dropped_for_resolution_flag"] = ((UNRESOLVED_MODE == "drop") & resolution_mapping_AGENTS["users_course_id"].isna()).astype("Int8")
key_resolution_audit_AGENTS = pd.DataFrame([{"metric": "total_student_user_course_pairs", "value": len(resolution_mapping_AGENTS)}, {"metric": "resolved_pairs", "value": int(resolution_mapping_AGENTS["users_course_id_resolved_flag"].sum())}, {"metric": "unresolved_pairs", "value": int(resolution_mapping_AGENTS["users_course_id"].isna().sum())}, {"metric": "resolved_share", "value": float(resolution_mapping_AGENTS["users_course_id_resolved_flag"].mean())}, {"metric": "conflict_user_course_pairs_removed", "value": len(conflict_user_course)}, {"metric": "conflict_user_ucid_pairs_removed", "value": len(conflict_user_ucid)}, {"metric": "observed_student_users_course_id", "value": all_user_ucid["users_course_id"].nunique()}])
unresolved_reason_AGENTS = resolution_mapping_AGENTS.loc[resolution_mapping_AGENTS["users_course_id"].isna(), ["user_id", "course_id"]].merge(user_ucid_count, on="user_id", how="left")
unresolved_reason_AGENTS["reason_bucket"] = np.where(unresolved_reason_AGENTS["observed_users_course_id_count"].isna(), "no_observed_users_course_id_for_user", "ambiguous_or_no_reliable_signal")
unresolved_reason_summary_AGENTS = unresolved_reason_AGENTS["reason_bucket"].value_counts().rename_axis("reason_bucket").reset_index(name="pair_count")
unresolved_course_summary_AGENTS = unresolved_reason_AGENTS.groupby("course_id").size().rename("unresolved_pair_count").reset_index().sort_values("unresolved_pair_count", ascending=False)
display(key_resolution_audit_AGENTS)
display(unresolved_reason_summary_AGENTS)
display(unresolved_course_summary_AGENTS.head(25))
plot_bar(resolution_mapping_AGENTS["users_course_id_resolved_flag"].value_counts().sort_index(), "Resolution status", "0 unresolved, 1 resolved", "pairs")


## 6. Resolution coverage and base entity

This block proves that the restored key gives high coverage for `user_lessons` and `wk_users_courses_actions`. `user_access_histories` total coverage is lower because the table has ids outside observed student activity scope.


In [ ]:
resolved_ids = set(resolution_mapping_AGENTS["users_course_id"].dropna().astype("int64"))
observed_student_ucids = set(all_user_ucid["users_course_id"].dropna().astype("int64"))
def coverage_for_users_course_id(table_name: str, df: pd.DataFrame) -> dict:
    ids_total = set(df["users_course_id"].dropna().astype("int64"))
    rows_total = len(df)
    covered_rows = int(df["users_course_id"].isin(resolved_ids).sum())
    observed_ids = ids_total & observed_student_ucids
    return {"table": table_name, "unique_ids_total": len(ids_total), "covered_unique_ids_total": len(ids_total & resolved_ids), "covered_unique_share_total": len(ids_total & resolved_ids) / len(ids_total) if ids_total else 0.0, "rows_total": rows_total, "covered_rows_total": covered_rows, "covered_row_share_total": covered_rows / rows_total if rows_total else 0.0, "observed_scope_unique_ids": len(observed_ids), "observed_scope_covered_unique_ids": len(observed_ids & resolved_ids), "observed_scope_covered_unique_share": len(observed_ids & resolved_ids) / len(observed_ids) if observed_ids else 0.0}
resolution_table_coverage_AGENTS = pd.DataFrame([coverage_for_users_course_id("user_lessons", dfs["user_lessons"].rename(columns={"user_lesson_row_id":"unused"})), coverage_for_users_course_id("wk_users_courses_actions", dfs["wk_users_courses_actions"]), coverage_for_users_course_id("user_access_histories", dfs["user_access_histories"])])
display(resolution_table_coverage_AGENTS)
plot_bar(resolution_table_coverage_AGENTS.set_index("table")["covered_unique_share_total"], "Resolved id coverage by table", "table", "unique id coverage")
base_entity_AGENTS = dfs["users_courses"].dropna(subset=["user_id", "course_id"]).drop_duplicates(subset=["user_id", "course_id"]).merge(resolution_mapping_AGENTS, on=["user_id", "course_id"], how="left", validate="one_to_one")
if UNRESOLVED_MODE == "drop":
    base_entity_AGENTS = base_entity_AGENTS.loc[base_entity_AGENTS["users_course_id_resolved_flag"].eq(1)].copy()
base_entity_audit_AGENTS = pd.DataFrame([{"metric": "base_rows_after_policy", "value": len(base_entity_AGENTS)}, {"metric": "duplicate_user_course", "value": int(base_entity_AGENTS.duplicated(["user_id", "course_id"]).sum())}, {"metric": "resolved_share_after_policy", "value": float(base_entity_AGENTS["users_course_id_resolved_flag"].mean())}])
display(base_entity_audit_AGENTS)


## 7. Feature engineering by source

Each feature block is aggregated to a safe join grain: `course_id`, `users_course_id`, or `user_id`. Raw ids are not used as predictive features.


### 7.1 Course-level features from `lessons`

This block keeps compact course structure features. It does not replace `users_courses` course-level fields because source EDA showed mismatches.


In [ ]:
lessons = dfs["lessons"].copy()
lessons_agg_AGENTS = lessons.groupby("course_id").agg(course_lessons_count=("course_id", "size"), course_task_expected_share=("task_expected", "mean"), course_conspect_expected_share=("conspect_expected", "mean"), course_survival_training_share=("wk_survival_training_expected", "mean"), course_scratch_enabled_share=("wk_scratch_playground_enabled", "mean"), course_attendance_tracking_share=("wk_attendance_tracking_enabled", "mean"), course_lessons_wk_max_points_sum=("wk_max_points", "sum"), course_lessons_wk_task_count_sum=("wk_task_count", "sum"), course_video_duration_sum=("wk_video_duration", "sum"), course_video_duration_mean=("wk_video_duration", "mean")).reset_index()
display_head(lessons_agg_AGENTS)


### 7.2 Lesson progress features from `user_lessons`

These features are course-specific after the restored key because they are aggregated by `users_course_id`.


In [ ]:
ul = dfs["user_lessons"].copy()
user_lessons_features_AGENTS = ul.groupby("users_course_id").agg(ul_rows=("lesson_id", "size"), ul_lesson_nunique=("lesson_id", "nunique"), ul_group_nunique=("group_id", "nunique"), ul_video_visited_sum=("video_visited", "sum"), ul_video_viewed_sum=("video_viewed", "sum"), ul_translation_visited_sum=("translation_visited", "sum"), ul_solved_lessons_sum=("solved", "sum"), ul_solved_tasks_sum=("solved_tasks_count", "sum"), ul_wk_solved_task_sum=("wk_solved_task_count", "sum"), ul_points_sum=("wk_points", "sum"), ul_points_mean=("wk_points", "mean")).reset_index()
user_lessons_features_AGENTS["ul_video_visited_share"] = safe_div(user_lessons_features_AGENTS["ul_video_visited_sum"], user_lessons_features_AGENTS["ul_rows"])
user_lessons_features_AGENTS["ul_translation_visited_share"] = safe_div(user_lessons_features_AGENTS["ul_translation_visited_sum"], user_lessons_features_AGENTS["ul_rows"])
user_lessons_features_AGENTS["ul_solved_lesson_share"] = safe_div(user_lessons_features_AGENTS["ul_solved_lessons_sum"], user_lessons_features_AGENTS["ul_rows"])
display_head(user_lessons_features_AGENTS)


### 7.3 Course action temporal features from `wk_users_courses_actions`

This block improves the weaker single aggregate from the old EDA by adding action diversity, first/last action, active days, activity span and max/mean gaps.


In [ ]:
wka = dfs["wk_users_courses_actions"].dropna(subset=["users_course_id"]).copy()
wka_base_agg = wka.groupby("users_course_id").agg(wka_event_count=("action_clean", "size"), wka_action_nunique=("action_clean", "nunique"), wka_lesson_nunique=("lesson_id", "nunique"), wka_first_action_at=("created_at", "min"), wka_last_action_at=("created_at", "max"), wka_active_days=("created_at", lambda s: s.dt.normalize().nunique())).reset_index()
wka_counts = pd.crosstab(wka["users_course_id"], wka["action_clean"]).add_prefix("wka_action_count_").reset_index()
wka_features_AGENTS = wka_base_agg.merge(wka_counts, on="users_course_id", how="left", validate="one_to_one")
wka_features_AGENTS["wka_activity_span_days"] = (wka_features_AGENTS["wka_last_action_at"] - wka_features_AGENTS["wka_first_action_at"]).dt.days
for col in [c for c in wka_features_AGENTS.columns if c.startswith("wka_action_count_")]:
    wka_features_AGENTS[col.replace("wka_action_count_", "wka_action_share_")] = safe_div(wka_features_AGENTS[col], wka_features_AGENTS["wka_event_count"])
wka_sorted = wka.dropna(subset=["created_at"]).sort_values(["users_course_id", "created_at"])
wka_sorted["wka_gap_days"] = wka_sorted.groupby("users_course_id")["created_at"].diff().dt.total_seconds() / 86400
wka_gap_features = wka_sorted.groupby("users_course_id").agg(wka_gap_days_mean=("wka_gap_days", "mean"), wka_gap_days_max=("wka_gap_days", "max")).reset_index()
wka_features_AGENTS = wka_features_AGENTS.merge(wka_gap_features, on="users_course_id", how="left", validate="one_to_one")
display_head(wka_features_AGENTS)


### 7.4 Access features from `user_access_histories`

Access windows can explain non-standard completion and churn patterns. This block is joined by restored `users_course_id`.


In [ ]:
access = dfs["user_access_histories"].copy()
access_agg_AGENTS = access.groupby("users_course_id").agg(access_rows=("users_course_id", "size"), access_started_min=("access_started_at", "min"), access_started_max=("access_started_at", "max"), access_expired_min=("access_expired_at", "min"), access_expired_max=("access_expired_at", "max"), access_interval_days_sum=("access_interval_days", "sum"), access_interval_days_max=("access_interval_days", "max"), access_activator_nunique=("activator_clean", "nunique")).reset_index()
access_counts = pd.crosstab(access["users_course_id"], access["activator_clean"]).add_prefix("access_activator_count_").reset_index()
access_features_AGENTS = access_agg_AGENTS.merge(access_counts, on="users_course_id", how="left", validate="one_to_one")
access_features_AGENTS["access_window_days"] = (access_features_AGENTS["access_expired_max"] - access_features_AGENTS["access_started_min"]).dt.days
display_head(access_features_AGENTS)


### 7.5 Auxiliary activity features from `user_activity_histories`

This table is not a replacement for WKA. It is joined through `user_lesson_id -> user_lessons.user_lesson_row_id` and used only as auxiliary activity evidence with explicit bridge audit.


In [ ]:
ul_bridge = dfs["user_lessons"][["user_lesson_row_id", "users_course_id"]].dropna().drop_duplicates(subset=["user_lesson_row_id"])
uahist = dfs["user_activity_histories"].copy()
uahist_joined = uahist.merge(ul_bridge, left_on="user_lesson_id", right_on="user_lesson_row_id", how="left")
user_activity_bridge_audit_AGENTS = pd.DataFrame([{"metric": "uahist_rows", "value": len(uahist_joined)}, {"metric": "uahist_rows_joined_to_user_lessons", "value": int(uahist_joined["users_course_id"].notna().sum())}, {"metric": "uahist_joined_row_share", "value": float(uahist_joined["users_course_id"].notna().mean())}, {"metric": "uahist_unique_user_lesson_id", "value": uahist["user_lesson_id"].nunique()}])
uahist_joined = uahist_joined.dropna(subset=["users_course_id"]).copy()
uahist_base = uahist_joined.groupby("users_course_id").agg(uahist_event_count=("action_clean", "size"), uahist_action_nunique=("action_clean", "nunique"), uahist_first_at=("created_at", "min"), uahist_last_at=("created_at", "max")).reset_index()
uahist_counts = pd.crosstab(uahist_joined["users_course_id"], uahist_joined["action_clean"]).add_prefix("uahist_action_count_").reset_index()
user_activity_features_AGENTS = uahist_base.merge(uahist_counts, on="users_course_id", how="left", validate="one_to_one")
display(user_activity_bridge_audit_AGENTS)
display_head(user_activity_features_AGENTS)


### 7.6 User profile features from `users`

These are user-level enrichment features. They are not course-specific but can help model general engagement background.


In [ ]:
users = dfs["users"].copy()
users_features_AGENTS = users[["user_id", "sign_in_count", "grade_id", "timezone", "d_wk_school_id", "d_wk_municipal_id", "d_wk_region_id", "wk_gender", "user_subscribed_flag"]].drop_duplicates(subset=["user_id"]).copy()
display_head(users_features_AGENTS)


### 7.7 Training features from `user_trainings`

Training records have no validated course key in raw data, so this block is user-level. It is especially relevant for the success proxy because marks `>= 4` are part of the business definition.


In [ ]:
train = dfs["user_trainings"].copy()
train_agg = train.groupby("user_id").agg(train_rows=("training_id", "size"), train_unique_trainings=("training_id", "nunique"), train_solved_tasks_sum=("solved_tasks_count", "sum"), train_earned_points_sum=("earned_points", "sum"), train_submitted_answers_sum=("submitted_answers_count", "sum"), train_attempts_sum=("attempts", "sum"), train_mark_mean=("mark", "mean"), train_mark_ge4_count=("mark_ge4_flag", "sum"), train_first_started_at=("started_at", "min"), train_last_finished_at=("finished_at", "max"), train_duration_min_mean=("training_duration_min", "mean")).reset_index()
train_type_counts = pd.crosstab(train["user_id"], train["type_clean"]).add_prefix("train_type_count_").reset_index()
train_state_counts = pd.crosstab(train["user_id"], train["state_clean"]).add_prefix("train_state_count_").reset_index()
training_features_AGENTS = train_agg.merge(train_type_counts, on="user_id", how="left", validate="one_to_one").merge(train_state_counts, on="user_id", how="left", validate="one_to_one")
display_head(training_features_AGENTS)


### 7.8 Media session features from `wk_media_view_sessions`

Media features are user-level only. They support engagement hypotheses but should not be interpreted as exact course-level behavior.


In [ ]:
media = dfs["wk_media_view_sessions"].copy()
media_agg = media.groupby("user_id").agg(media_sessions=("resource_type", "size"), media_segments_total_sum=("segments_total", "sum"), media_segments_viewed_sum=("viewed_segments_count", "sum"), media_view_ratio_mean=("media_view_ratio_row", "mean"), media_first_started_at=("started_at", "min"), media_last_started_at=("started_at", "max"), media_kind_nunique=("kind_clean", "nunique")).reset_index()
media_kind_counts = pd.crosstab(media["user_id"], media["kind_clean"]).add_prefix("media_kind_count_").reset_index()
media_features_AGENTS = media_agg.merge(media_kind_counts, on="user_id", how="left", validate="one_to_one")
media_features_AGENTS["media_viewed_segments_share"] = safe_div(media_features_AGENTS["media_segments_viewed_sum"], media_features_AGENTS["media_segments_total_sum"])
display_head(media_features_AGENTS)


### 7.9 Award features from badges

Awards are user-level motivation proxies. The old EDA noted that some badge types may be mutually exclusive; here we keep compact counts and levels.


In [ ]:
award_ref = dfs["award_badges"]
award_events = dfs["user_award_badges"].merge(award_ref[["award_badge_id", "level", "quota", "special", "award_name_clean"]], on="award_badge_id", how="left")
award_agg = award_events.groupby("user_id").agg(badge_rows=("award_badge_id", "size"), badge_unique_count=("award_badge_id", "nunique"), badge_level_max=("level", "max"), badge_level_mean=("level", "mean"), badge_special_sum=("special", "sum"), badge_first_at=("created_at", "min"), badge_last_at=("created_at", "max")).reset_index()
award_name_counts = pd.crosstab(award_events["user_id"], award_events["award_name_clean"]).add_prefix("badge_name_count_").reset_index()
award_features_AGENTS = award_agg.merge(award_name_counts, on="user_id", how="left", validate="one_to_one")
display_head(award_features_AGENTS)


### 7.10 User answers chunk aggregation

`user_answers` is large and has no reliable course key in the available raw columns. It is therefore processed in chunks and used as user-level task-engagement enrichment.


In [ ]:
def aggregate_user_answers_chunks(path: Path, chunk_size: int, teacher_ids: set) -> pd.DataFrame:
    parts = []
    usecols = ["user_id", "attempts", "solved", "points", "max_attempts", "skipped", "resource_type", "submitted_at", "wk_partial_answer", "performance", "async_check_status"]
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunk_size, encoding="utf-8", low_memory=False):
        chunk["user_id"] = to_int_id(chunk["user_id"])
        chunk = chunk.loc[~chunk["user_id"].isin(teacher_ids)].copy()
        chunk["attempts"] = pd.to_numeric(chunk["attempts"], errors="coerce")
        chunk["max_attempts"] = pd.to_numeric(chunk["max_attempts"], errors="coerce")
        chunk["points"] = pd.to_numeric(chunk["points"], errors="coerce")
        chunk["performance"] = pd.to_numeric(chunk["performance"], errors="coerce")
        chunk["solved_flag"] = to_bool01(chunk["solved"])
        chunk["skipped_flag"] = to_bool01(chunk["skipped"])
        chunk["partial_flag"] = to_bool01(chunk["wk_partial_answer"])
        chunk["submitted_at"] = pd.to_datetime(chunk["submitted_at"], errors="coerce")
        chunk["resource_clean"] = chunk["resource_type"].astype("string").map(clean_name)
        chunk["ua_rows_tmp"] = 1
        chunk["performance_nonnull"] = chunk["performance"].notna().astype("Int8")
        base = chunk.groupby("user_id").agg(answer_rows=("ua_rows_tmp", "sum"), answer_solved_sum=("solved_flag", "sum"), answer_skipped_sum=("skipped_flag", "sum"), answer_partial_sum=("partial_flag", "sum"), answer_attempts_sum=("attempts", "sum"), answer_max_attempts_sum=("max_attempts", "sum"), answer_points_sum=("points", "sum"), answer_performance_sum=("performance", "sum"), answer_performance_count=("performance_nonnull", "sum"), answer_first_submitted_at=("submitted_at", "min"), answer_last_submitted_at=("submitted_at", "max")).reset_index()
        resource_counts = pd.crosstab(chunk["user_id"], chunk["resource_clean"]).add_prefix("answer_resource_count_").reset_index()
        parts.append(base.merge(resource_counts, on="user_id", how="left"))
    stacked = pd.concat(parts, ignore_index=True)
    sum_cols = [c for c in stacked.columns if c not in ["user_id", "answer_first_submitted_at", "answer_last_submitted_at"]]
    stacked[sum_cols] = stacked[sum_cols].fillna(0)
    out_sum = stacked.groupby("user_id")[sum_cols].sum().reset_index()
    out_dates = stacked.groupby("user_id").agg(answer_first_submitted_at=("answer_first_submitted_at", "min"), answer_last_submitted_at=("answer_last_submitted_at", "max")).reset_index()
    out = out_sum.merge(out_dates, on="user_id", how="left", validate="one_to_one")
    out["answer_solved_share"] = safe_div(out["answer_solved_sum"], out["answer_rows"])
    out["answer_skipped_share"] = safe_div(out["answer_skipped_sum"], out["answer_rows"])
    out["answer_partial_share"] = safe_div(out["answer_partial_sum"], out["answer_rows"])
    out["answer_attempts_per_answer"] = safe_div(out["answer_attempts_sum"], out["answer_rows"])
    out["answer_points_per_answer"] = safe_div(out["answer_points_sum"], out["answer_rows"])
    out["answer_performance_mean"] = safe_div(out["answer_performance_sum"], out["answer_performance_count"])
    return out
answer_features_AGENTS = aggregate_user_answers_chunks(DATA_DIR / "user_answers.csv", USER_ANSWERS_CHUNK_SIZE, teacher_ids)
display(pd.DataFrame([{"feature_block": "answer_features", "rows": len(answer_features_AGENTS), "cols": answer_features_AGENTS.shape[1]}]))
display_head(answer_features_AGENTS)


## 8. Hypotheses block

Each hypothesis is separated and produces metrics, tables and plots. The interpretation is stored in `hypothesis_results_AGENTS` and in the final `Hipotises_AGENTS.md` export.


### H1. Course-level lessons aggregation is useful but not perfect

Why it matters: course-level metadata helps normalize student behavior, but source EDA showed that `lessons` fields do not always exactly match `users_courses`. Decision: use compact course features, not as authoritative replacements.


In [ ]:
uc_course_ref = dfs["users_courses"].dropna(subset=["course_id"]).groupby("course_id").agg(uc_wk_max_points_median=("wk_max_points", "median"), uc_wk_max_task_count_median=("wk_max_task_count", "median"), uc_wk_max_viewable_lessons_median=("wk_max_viewable_lessons", "median"), uc_course_rows=("course_id", "size")).reset_index()
course_validation_AGENTS = uc_course_ref.merge(lessons_agg_AGENTS, on="course_id", how="left")
course_validation_AGENTS["diff_max_points"] = course_validation_AGENTS["uc_wk_max_points_median"] - course_validation_AGENTS["course_lessons_wk_max_points_sum"]
course_validation_AGENTS["diff_task_count"] = course_validation_AGENTS["uc_wk_max_task_count_median"] - course_validation_AGENTS["course_lessons_wk_task_count_sum"]
course_validation_AGENTS["diff_lesson_count"] = course_validation_AGENTS["uc_wk_max_viewable_lessons_median"] - course_validation_AGENTS["course_lessons_count"]
course_validation_clean = course_validation_AGENTS.loc[course_validation_AGENTS["uc_course_rows"].ge(MIN_COURSE_ROWS_FOR_COURSE_AUDIT)].copy()
h1_metrics_AGENTS = pd.DataFrame([{"metric": "courses_total", "value": len(course_validation_AGENTS)}, {"metric": "courses_clean_min_rows", "value": len(course_validation_clean)}, {"metric": "max_points_exact_share", "value": float(course_validation_clean["diff_max_points"].eq(0).mean())}, {"metric": "task_count_exact_share", "value": float(course_validation_clean["diff_task_count"].eq(0).mean())}, {"metric": "lesson_count_exact_share", "value": float(course_validation_clean["diff_lesson_count"].eq(0).mean())}, {"metric": "max_points_abs_diff_mean", "value": float(course_validation_clean["diff_max_points"].abs().mean())}, {"metric": "task_count_abs_diff_mean", "value": float(course_validation_clean["diff_task_count"].abs().mean())}, {"metric": "lesson_count_abs_diff_mean", "value": float(course_validation_clean["diff_lesson_count"].abs().mean())}])
display(h1_metrics_AGENTS)
display(course_validation_clean.sort_values("diff_lesson_count", key=lambda s: s.abs(), ascending=False).head(20))
plt.figure(figsize=(8, 3)); course_validation_clean[["diff_max_points", "diff_task_count", "diff_lesson_count"]].clip(-50, 50).hist(bins=30, figsize=(10, 3)); plt.suptitle("H1 residual distributions clipped"); plt.tight_layout(); plt.show()


### H2. Restored key bridge is sufficient for main course-activity tables

Why it matters: final features require joining `users_courses` with activity tables. Decision: use restored `users_course_id`, keep/drop unresolved rows through config, and keep audit flags.


In [ ]:
h2_metrics_AGENTS = pd.concat([key_resolution_audit_AGENTS.assign(section="resolution"), resolution_table_coverage_AGENTS.melt(id_vars="table", var_name="metric", value_name="value").assign(section=lambda x: "coverage_" + x["table"].astype(str))[['section','metric','value']]], ignore_index=True, sort=False)
display(h2_metrics_AGENTS.head(60))
display(method_acceptance_audit_AGENTS)
display(unresolved_reason_summary_AGENTS)
display(unresolved_course_summary_AGENTS.head(30))
plt.figure(figsize=(7, 3)); unresolved_course_summary_AGENTS.head(15).set_index("course_id")["unresolved_pair_count"].plot(kind="bar"); plt.title("H2 unresolved pairs by course top 15"); plt.xlabel("course_id"); plt.ylabel("unresolved pairs"); plt.tight_layout(); plt.show()


### H3. `user_activity_histories` partially overlaps WKA and is auxiliary

Why it matters: source EDA considered excluding `user_activity_histories`. Here we check bridge coverage and action/timestamp overlap. Decision: use it only as auxiliary features because WKA is the main course-action log.


In [ ]:
action_map_h3 = {"visit_video": ["visit_video"], "visit_translation": ["visit_translation"], "show_conspect": ["visit_preparation_material"]}
uah_for_h3 = dfs["user_activity_histories"].copy()
wka_for_h3 = dfs["wk_users_courses_actions"].copy()
h3_time_rows = []
h3_exact_rows = []
for ua_action, wka_actions in action_map_h3.items():
    ua_sub = uah_for_h3.loc[uah_for_h3["action_clean"].eq(clean_name(ua_action)), ["created_at"]].dropna()
    wka_sub = wka_for_h3.loc[wka_for_h3["action_clean"].isin([clean_name(a) for a in wka_actions]), ["created_at"]].dropna()
    h3_time_rows.append({"ua_action": ua_action, "wka_actions": "|".join(wka_actions), "uah_first_at": ua_sub["created_at"].min(), "uah_last_at": ua_sub["created_at"].max(), "wka_first_at": wka_sub["created_at"].min(), "wka_last_at": wka_sub["created_at"].max(), "uah_events": len(ua_sub), "wka_events": len(wka_sub)})
    ua_counts = ua_sub.groupby("created_at").size().rename("uah_count").reset_index()
    wka_counts_tmp = wka_sub.groupby("created_at").size().rename("wka_count").reset_index()
    cmp = ua_counts.merge(wka_counts_tmp, on="created_at", how="outer").fillna(0)
    cmp["covered_count"] = np.minimum(cmp["uah_count"], cmp["wka_count"])
    h3_exact_rows.append({"ua_action": ua_action, "uah_total_events": int(cmp["uah_count"].sum()), "wka_total_events": int(cmp["wka_count"].sum()), "covered_events_exact_ts": int(cmp["covered_count"].sum()), "coverage_ratio_exact_ts": float(cmp["covered_count"].sum() / cmp["uah_count"].sum()) if cmp["uah_count"].sum() else np.nan})
h3_time_bounds_AGENTS = pd.DataFrame(h3_time_rows)
h3_exact_overlap_AGENTS = pd.DataFrame(h3_exact_rows)
display(user_activity_bridge_audit_AGENTS)
display(h3_time_bounds_AGENTS)
display(h3_exact_overlap_AGENTS)
plt.figure(figsize=(6, 3)); h3_exact_overlap_AGENTS.set_index("ua_action")["coverage_ratio_exact_ts"].plot(kind="bar"); plt.title("H3 exact timestamp overlap"); plt.xlabel("UAH action"); plt.ylabel("coverage ratio"); plt.tight_layout(); plt.show()


### H4. Activity intensity is related to progress and weak success

Why it matters: churn and success are behavior-driven. WKA counts, active days and gaps should separate stronger vs weaker progress patterns.


In [ ]:
hyp_frame = base_entity_AGENTS[["user_id", "course_id", "users_course_id", "users_course_id_resolved_flag", "uc_points_ratio", "uc_is_completed_flag", "access_finished_at"]].copy()
hyp_frame = hyp_frame.merge(wka_features_AGENTS[["users_course_id", "wka_event_count", "wka_first_action_at", "wka_last_action_at", "wka_active_days", "wka_gap_days_max"]], on="users_course_id", how="left")
hyp_frame = hyp_frame.merge(user_lessons_features_AGENTS[["users_course_id", "ul_lesson_nunique", "ul_solved_lesson_share", "ul_points_sum"]], on="users_course_id", how="left")
hyp_frame = hyp_frame.merge(training_features_AGENTS[["user_id", "train_rows", "train_mark_mean", "train_mark_ge4_count"]], on="user_id", how="left")
hyp_frame["days_last_action_to_access_end"] = (hyp_frame["access_finished_at"] - hyp_frame["wka_last_action_at"]).dt.days
hyp_frame["churn_proxy_30d"] = (hyp_frame["users_course_id_resolved_flag"].eq(1) & hyp_frame["wka_event_count"].fillna(0).gt(0) & hyp_frame["uc_is_completed_flag"].eq(0) & hyp_frame["days_last_action_to_access_end"].ge(TARGET_INACTIVITY_DAYS)).astype("Int8")
hyp_frame["success_proxy_weak"] = (hyp_frame["uc_points_ratio"].ge(0.5) & hyp_frame["train_mark_ge4_count"].fillna(0).ge(4)).astype("Int8")
hyp_frame["wka_event_bin"] = pd.qcut(hyp_frame["wka_event_count"].rank(method="first"), q=5, duplicates="drop")
h4_bin_summary_AGENTS = hyp_frame.groupby("wka_event_bin", dropna=False).agg(rows=("user_id", "size"), mean_progress=("uc_points_ratio", "mean"), success_proxy_rate=("success_proxy_weak", "mean"), churn_proxy_rate=("churn_proxy_30d", "mean"), mean_active_days=("wka_active_days", "mean"), mean_max_gap=("wka_gap_days_max", "mean")).reset_index()
display(h4_bin_summary_AGENTS)
plt.figure(figsize=(8, 3)); h4_bin_summary_AGENTS.set_index("wka_event_bin")[["mean_progress", "success_proxy_rate", "churn_proxy_rate"]].plot(kind="bar", figsize=(10, 4)); plt.title("H4 activity bins vs outcomes"); plt.xlabel("wka event bin"); plt.tight_layout(); plt.show()


### H5. Last activity before access end is a churn-risk signal

Why it matters: the business churn definition explicitly depends on stopping activity before course end/access end.


In [ ]:
hyp_frame["recency_bin"] = pd.cut(hyp_frame["days_last_action_to_access_end"], bins=[-9999, -1, 0, 7, 14, 30, 60, 120, 9999], labels=["after_end", "same_day", "1_7", "8_14", "15_30", "31_60", "61_120", "120_plus"])
h5_recency_summary_AGENTS = hyp_frame.groupby("recency_bin", dropna=False).agg(rows=("user_id", "size"), churn_proxy_rate=("churn_proxy_30d", "mean"), mean_progress=("uc_points_ratio", "mean"), completed_rate=("uc_is_completed_flag", "mean")).reset_index()
display(h5_recency_summary_AGENTS)
plt.figure(figsize=(8, 3)); h5_recency_summary_AGENTS.set_index("recency_bin")["churn_proxy_rate"].plot(kind="bar"); plt.title("H5 churn proxy by recency bin"); plt.xlabel("days last action to access end"); plt.ylabel("churn proxy rate"); plt.tight_layout(); plt.show()


### H6. Access windows and repeated access can mark non-standard learning paths

Why it matters: repeated or long access may indicate extensions, interruptions or non-standard progression. Decision: include access counts and durations as interpretable features.


In [ ]:
h6_frame = hyp_frame.merge(access_features_AGENTS[["users_course_id", "access_rows", "access_window_days", "access_activator_nunique"]], on="users_course_id", how="left")
h6_frame["access_rows_bin"] = pd.cut(h6_frame["access_rows"].fillna(0), bins=[-1, 0, 1, 2, 5, 999], labels=["no_access", "one", "two", "three_to_five", "five_plus"])
h6_access_summary_AGENTS = h6_frame.groupby("access_rows_bin", dropna=False).agg(rows=("user_id", "size"), mean_progress=("uc_points_ratio", "mean"), churn_proxy_rate=("churn_proxy_30d", "mean"), mean_access_window_days=("access_window_days", "mean")).reset_index()
display(h6_access_summary_AGENTS)
plt.figure(figsize=(8, 3)); h6_access_summary_AGENTS.set_index("access_rows_bin")[["mean_progress", "churn_proxy_rate"]].plot(kind="bar", figsize=(9, 4)); plt.title("H6 access rows vs progress/churn"); plt.xlabel("access rows bin"); plt.tight_layout(); plt.show()


### H7. User-level engagement enrichments are useful but not course-specific

Why it matters: media, badges and answers are user-level signals. They can improve baseline quality, but interpretation must be careful because they are not course-specific joins.


In [ ]:
h7_frame = hyp_frame.merge(media_features_AGENTS[["user_id", "media_sessions", "media_viewed_segments_share"]], on="user_id", how="left").merge(award_features_AGENTS[["user_id", "badge_rows", "badge_unique_count"]], on="user_id", how="left").merge(answer_features_AGENTS[["user_id", "answer_rows", "answer_solved_share", "answer_attempts_per_answer", "answer_performance_mean"]], on="user_id", how="left")
engagement_cols = ["media_sessions", "media_viewed_segments_share", "badge_rows", "badge_unique_count", "answer_rows", "answer_solved_share", "answer_attempts_per_answer", "answer_performance_mean"]
h7_corr_rows = []
for col in engagement_cols:
    h7_corr_rows.append({"feature": col, "corr_with_progress": float(h7_frame[[col, "uc_points_ratio"]].corr().iloc[0, 1]) if h7_frame[col].notna().sum() > 1 else np.nan, "nonnull_share": float(h7_frame[col].notna().mean())})
h7_user_engagement_corr_AGENTS = pd.DataFrame(h7_corr_rows).sort_values("corr_with_progress", key=lambda s: s.abs(), ascending=False)
display(h7_user_engagement_corr_AGENTS)
plt.figure(figsize=(8, 3)); h7_user_engagement_corr_AGENTS.set_index("feature")["corr_with_progress"].plot(kind="bar"); plt.title("H7 user-level feature correlation with progress"); plt.xlabel("feature"); plt.ylabel("corr"); plt.tight_layout(); plt.show()


### H8. Target proxy distributions and leakage note

Why it matters: target-related columns are useful for analysis, but post-outcome fields must be excluded from a safer ML feature matrix.


In [ ]:
target_proxy_summary_AGENTS = pd.DataFrame([{"target_proxy": "churn_proxy_30d", "nonnull_rows": int(hyp_frame["churn_proxy_30d"].notna().sum()), "positive_rate": float(hyp_frame["churn_proxy_30d"].dropna().mean())}, {"target_proxy": "success_proxy_weak", "nonnull_rows": int(hyp_frame["success_proxy_weak"].notna().sum()), "positive_rate": float(hyp_frame["success_proxy_weak"].dropna().mean())}])
display(target_proxy_summary_AGENTS)
plt.figure(figsize=(6, 3)); hyp_frame["churn_proxy_30d"].value_counts(dropna=False).sort_index().plot(kind="bar"); plt.title("H8 churn proxy distribution"); plt.xlabel("flag"); plt.ylabel("rows"); plt.tight_layout(); plt.show()
plt.figure(figsize=(6, 3)); hyp_frame["success_proxy_weak"].value_counts(dropna=False).sort_index().plot(kind="bar"); plt.title("H8 weak success proxy distribution"); plt.xlabel("flag"); plt.ylabel("rows"); plt.tight_layout(); plt.show()


### Hypothesis interpretation table

This table is the compact analytical summary used later in `Hipotises_AGENTS.md`.


In [ ]:
hypothesis_rows = []
for _, row in h1_metrics_AGENTS.iterrows():
    hypothesis_rows.append({"hypothesis": "H1_lessons_course_aggregation", "metric": row["metric"], "value": row["value"], "decision": "use_lessons_as_compact_course_features_with_audit"})
hypothesis_rows.extend([
    {"hypothesis": "H2_key_resolution", "metric": "resolved_share", "value": float(key_resolution_audit_AGENTS.loc[key_resolution_audit_AGENTS["metric"].eq("resolved_share"), "value"].iloc[0]), "decision": "use_restored_bridge_with_unresolved_policy"},
    {"hypothesis": "H3_user_activity_auxiliary", "metric": "uahist_joined_row_share", "value": float(user_activity_bridge_audit_AGENTS.loc[user_activity_bridge_audit_AGENTS["metric"].eq("uahist_joined_row_share"), "value"].iloc[0]), "decision": "use_as_auxiliary_not_replacement"},
    {"hypothesis": "H4_activity_intensity", "metric": "mean_wka_events_success_proxy_0", "value": float(hyp_frame.loc[hyp_frame["success_proxy_weak"].eq(0), "wka_event_count"].mean()), "decision": "use_wka_counts_active_days_gaps"},
    {"hypothesis": "H4_activity_intensity", "metric": "mean_wka_events_success_proxy_1", "value": float(hyp_frame.loc[hyp_frame["success_proxy_weak"].eq(1), "wka_event_count"].mean()), "decision": "use_wka_counts_active_days_gaps"},
    {"hypothesis": "H5_recency_churn", "metric": "churn_proxy_rate", "value": float(hyp_frame["churn_proxy_30d"].mean()), "decision": "use_recency_features_and_target_proxy"},
    {"hypothesis": "H6_access_windows", "metric": "mean_access_rows", "value": float(h6_frame["access_rows"].mean()), "decision": "use_access_count_and_duration_features"},
])
for _, row in h7_user_engagement_corr_AGENTS.iterrows():
    hypothesis_rows.append({"hypothesis": "H7_user_level_engagement", "metric": "corr_" + row["feature"], "value": row["corr_with_progress"], "decision": "use_as_user_level_enrichment"})
hypothesis_results_AGENTS = pd.DataFrame(hypothesis_rows)
display(hypothesis_results_AGENTS)


## 9. Join strategy and final merge

Join keys and expected cardinality:
- `course_id`: base many rows to one course feature row;
- `users_course_id`: base many rows including unresolved `NA`, but resolved ids map to one feature row;
- `user_id`: base many rows to one user-level feature row.

The merge audit proves that no row explosion happened.


In [ ]:
merge_audit_rows = []
final_modeling_table_AGENTS = base_entity_AGENTS.copy()
feature_blocks = [("lessons_course_features", lessons_agg_AGENTS, ["course_id"]), ("user_lessons_features", user_lessons_features_AGENTS, ["users_course_id"]), ("wka_features", wka_features_AGENTS, ["users_course_id"]), ("access_features", access_features_AGENTS, ["users_course_id"]), ("user_activity_features", user_activity_features_AGENTS, ["users_course_id"]), ("users_features", users_features_AGENTS, ["user_id"]), ("training_features", training_features_AGENTS, ["user_id"]), ("media_features", media_features_AGENTS, ["user_id"]), ("award_features", award_features_AGENTS, ["user_id"]), ("answer_features", answer_features_AGENTS, ["user_id"])]
for name, block, keys in feature_blocks:
    final_modeling_table_AGENTS = merge_many_to_one(final_modeling_table_AGENTS, block, keys, name, merge_audit_rows)
merge_audit_AGENTS = pd.DataFrame(merge_audit_rows)
display(merge_audit_AGENTS)


### 9.1 Missing policy after merge

For resolved ids, missing course-activity counts are filled with zero when the semantic meaning is no observed event. For unresolved ids, course-activity fields remain `NA` because the bridge is unknown.


In [ ]:
resolved_mask = final_modeling_table_AGENTS["users_course_id_resolved_flag"].eq(1)
course_event_prefixes = ("ul_", "wka_", "access_", "uahist_")
course_event_numeric_cols = [c for c in final_modeling_table_AGENTS.columns if c.startswith(course_event_prefixes) and pd.api.types.is_numeric_dtype(final_modeling_table_AGENTS[c])]
for col in course_event_numeric_cols:
    final_modeling_table_AGENTS.loc[resolved_mask, col] = final_modeling_table_AGENTS.loc[resolved_mask, col].fillna(0)
user_level_prefixes = ("train_", "media_", "badge_", "answer_")
user_level_numeric_cols = [c for c in final_modeling_table_AGENTS.columns if c.startswith(user_level_prefixes) and pd.api.types.is_numeric_dtype(final_modeling_table_AGENTS[c])]
for col in user_level_numeric_cols:
    final_modeling_table_AGENTS[col] = final_modeling_table_AGENTS[col].fillna(0)
zero_series = pd.Series(0, index=final_modeling_table_AGENTS.index)
final_modeling_table_AGENTS["maybe_action_data_missing_flag"] = (final_modeling_table_AGENTS["users_course_id_resolved_flag"].eq(0) | (final_modeling_table_AGENTS.get("wka_event_count", zero_series).fillna(0).eq(0) & final_modeling_table_AGENTS.get("ul_rows", zero_series).fillna(0).eq(0))).astype("Int8")


### 9.2 Target-related columns and leakage notes

`target_churn_proxy_30d_flag` follows the case description in simplified form: activity existed, then no action for at least 30 days before access end, and course was not completed. `target_success_proxy_weak_flag` is a weak approximation because the exact 4 intermediate tests are not reliably course-keyed in raw data.


In [ ]:
final_modeling_table_AGENTS["days_last_action_to_access_end"] = (final_modeling_table_AGENTS["access_finished_at"] - final_modeling_table_AGENTS["wka_last_action_at"]).dt.days
final_modeling_table_AGENTS["target_had_course_activity_flag"] = (final_modeling_table_AGENTS["users_course_id_resolved_flag"].eq(1) & (final_modeling_table_AGENTS.get("wka_event_count", zero_series).fillna(0).gt(0) | final_modeling_table_AGENTS.get("ul_rows", zero_series).fillna(0).gt(0))).astype("Int8")
final_modeling_table_AGENTS["target_churn_proxy_30d_flag"] = (final_modeling_table_AGENTS["target_had_course_activity_flag"].eq(1) & final_modeling_table_AGENTS["uc_is_completed_flag"].eq(0) & final_modeling_table_AGENTS["days_last_action_to_access_end"].ge(TARGET_INACTIVITY_DAYS)).astype("Int8")
final_modeling_table_AGENTS.loc[final_modeling_table_AGENTS["users_course_id_resolved_flag"].eq(0), "target_churn_proxy_30d_flag"] = pd.NA
final_modeling_table_AGENTS["target_success_proxy_weak_flag"] = (final_modeling_table_AGENTS["uc_points_ratio"].ge(0.5) & final_modeling_table_AGENTS.get("train_mark_ge4_count", zero_series).fillna(0).ge(4)).astype("Int8")
leakage_note_AGENTS = pd.DataFrame([{"column": col, "leakage_risk": "target_or_post_outcome", "recommendation": "exclude_from_safe_feature_matrix"} for col in ["wk_course_completed_at", "uc_is_completed_flag", "target_churn_proxy_30d_flag", "target_success_proxy_weak_flag"] if col in final_modeling_table_AGENTS.columns])
display(leakage_note_AGENTS)


## 10. Final dataset audit

This audit checks row counts, grain uniqueness, merge coverage, missingness, target distributions and sanity plots.


In [ ]:
final_audit_AGENTS = pd.DataFrame([{"metric": "final_rows", "value": len(final_modeling_table_AGENTS)}, {"metric": "final_duplicate_user_course", "value": int(final_modeling_table_AGENTS.duplicated(["user_id", "course_id"]).sum())}, {"metric": "final_resolved_share", "value": float(final_modeling_table_AGENTS["users_course_id_resolved_flag"].mean())}, {"metric": "final_action_missing_share", "value": float(final_modeling_table_AGENTS["maybe_action_data_missing_flag"].mean())}, {"metric": "target_churn_proxy_30d_rate_nonnull", "value": float(final_modeling_table_AGENTS["target_churn_proxy_30d_flag"].dropna().mean())}, {"metric": "target_success_proxy_weak_rate", "value": float(final_modeling_table_AGENTS["target_success_proxy_weak_flag"].mean())}])
missingness_audit_AGENTS = missingness_table(final_modeling_table_AGENTS, top_n=250)
display(final_audit_AGENTS)
display(missingness_audit_AGENTS.head(60))
plot_bar(final_modeling_table_AGENTS["users_course_id_resolved_flag"].value_counts().sort_index(), "Final resolution flag", "0 unresolved, 1 resolved", "rows")
plt.figure(figsize=(6, 3)); final_modeling_table_AGENTS["uc_points_ratio"].clip(0, 1).hist(bins=30); plt.title("Final progress ratio"); plt.xlabel("uc_points_ratio"); plt.ylabel("rows"); plt.tight_layout(); plt.show()
plot_bar(final_modeling_table_AGENTS["target_churn_proxy_30d_flag"].value_counts(dropna=False).sort_index(), "Final churn proxy", "flag", "rows")
plot_bar(final_modeling_table_AGENTS["target_success_proxy_weak_flag"].value_counts(dropna=False).sort_index(), "Final weak success proxy", "flag", "rows")


## 11. Export artifacts

All exports use ASCII names and are written only to `data/AGENTS`. CSV is used because this environment has no parquet engine installed.


In [ ]:
source_overview_AGENTS.to_csv(OUTPUT_DIR / "source_overview_AGENTS.csv", index=False, encoding="utf-8")
duplicate_audit_AGENTS.to_csv(OUTPUT_DIR / "source_duplicate_audit_AGENTS.csv", index=False, encoding="utf-8")
teacher_filter_audit_AGENTS.to_csv(OUTPUT_DIR / "teacher_filter_audit_AGENTS.csv", index=False, encoding="utf-8")
resolution_mapping_AGENTS.to_csv(OUTPUT_DIR / "resolution_mapping_AGENTS.csv", index=False, encoding="utf-8", na_rep="NA")
key_resolution_audit_AGENTS.to_csv(OUTPUT_DIR / "key_resolution_audit_AGENTS.csv", index=False, encoding="utf-8")
resolution_table_coverage_AGENTS.to_csv(OUTPUT_DIR / "resolution_table_coverage_AGENTS.csv", index=False, encoding="utf-8")
unresolved_reason_summary_AGENTS.to_csv(OUTPUT_DIR / "unresolved_reason_summary_AGENTS.csv", index=False, encoding="utf-8")
unresolved_course_summary_AGENTS.to_csv(OUTPUT_DIR / "unresolved_course_summary_AGENTS.csv", index=False, encoding="utf-8")
user_activity_bridge_audit_AGENTS.to_csv(OUTPUT_DIR / "user_activity_bridge_audit_AGENTS.csv", index=False, encoding="utf-8")
h1_metrics_AGENTS.to_csv(OUTPUT_DIR / "h1_course_validation_metrics_AGENTS.csv", index=False, encoding="utf-8")
h3_time_bounds_AGENTS.to_csv(OUTPUT_DIR / "h3_uah_wka_time_bounds_AGENTS.csv", index=False, encoding="utf-8")
h3_exact_overlap_AGENTS.to_csv(OUTPUT_DIR / "h3_uah_wka_exact_overlap_AGENTS.csv", index=False, encoding="utf-8")
h4_bin_summary_AGENTS.to_csv(OUTPUT_DIR / "h4_activity_bin_summary_AGENTS.csv", index=False, encoding="utf-8")
h5_recency_summary_AGENTS.to_csv(OUTPUT_DIR / "h5_recency_summary_AGENTS.csv", index=False, encoding="utf-8")
h6_access_summary_AGENTS.to_csv(OUTPUT_DIR / "h6_access_summary_AGENTS.csv", index=False, encoding="utf-8")
h7_user_engagement_corr_AGENTS.to_csv(OUTPUT_DIR / "h7_user_engagement_corr_AGENTS.csv", index=False, encoding="utf-8")
target_proxy_summary_AGENTS.to_csv(OUTPUT_DIR / "target_proxy_summary_AGENTS.csv", index=False, encoding="utf-8")
hypothesis_results_AGENTS.to_csv(OUTPUT_DIR / "hypothesis_results_AGENTS.csv", index=False, encoding="utf-8")
merge_audit_AGENTS.to_csv(OUTPUT_DIR / "merge_audit_AGENTS.csv", index=False, encoding="utf-8")
missingness_audit_AGENTS.to_csv(OUTPUT_DIR / "missingness_audit_AGENTS.csv", index=False, encoding="utf-8")
final_audit_AGENTS.to_csv(OUTPUT_DIR / "final_audit_AGENTS.csv", index=False, encoding="utf-8")
leakage_note_AGENTS.to_csv(OUTPUT_DIR / "feature_leakage_notes_AGENTS.csv", index=False, encoding="utf-8")
final_modeling_table_AGENTS.to_csv(OUTPUT_DIR / "final_modeling_table_AGENTS.csv", index=False, encoding="utf-8", na_rep="NA")
feature_dictionary_AGENTS = pd.DataFrame([{"column": col, "dtype": str(final_modeling_table_AGENTS[col].dtype), "source_hint": col.split("_")[0], "note": "generated_by_EDA_AGENT_pipeline"} for col in final_modeling_table_AGENTS.columns])
feature_dictionary_AGENTS.to_csv(OUTPUT_DIR / "feature_dictionary_AGENTS.csv", index=False, encoding="utf-8")

hyp_md = """# Hipotises_AGENTS

## Goal
Keep only hypotheses that were checked and used in final EDA or feature engineering.

## Confirmed hypotheses

1. H1_lessons_course_aggregation
- Check: compare sums/counts from `lessons` with canonical medians in `users_courses`.
- Result: useful but not perfect, especially for lesson counts.
- Consequence: use compact course features with audit, do not overwrite `users_courses`.

2. H2_key_resolution
- Check: restore `(user_id, course_id) -> users_course_id`, inspect conflicts and table coverage.
- Result: bridge covers main observed activity ids very well; unresolved rows are controlled by config.
- Consequence: all course-activity joins use restored `users_course_id`.

3. H3_user_activity_auxiliary
- Check: bridge `user_activity_histories.user_lesson_id` to `user_lessons.user_lesson_row_id` and compare overlap with WKA actions.
- Result: useful partial auxiliary source, not a replacement for WKA.
- Consequence: include auxiliary UAH action counts only with audit.

4. H4_activity_intensity
- Check: WKA event bins vs progress and weak success proxy.
- Result: activity intensity separates behavior patterns.
- Consequence: use WKA counts, active days, spans and gaps.

5. H5_recency_churn
- Check: last action to access end bins vs churn proxy.
- Result: recency is aligned with churn definition.
- Consequence: use recency and target proxy columns.

6. H6_access_windows
- Check: access row bins vs progress and churn proxy.
- Result: access history is useful as non-standard path context.
- Consequence: use access counts and duration features.

7. H7_user_level_engagement
- Check: user-level media, badge and answer correlations with progress.
- Result: useful as enrichment, but not course-specific.
- Consequence: join by `user_id` and interpret carefully.

## Rejected or limited hypotheses

- Raw ids are not used as predictive features.
- `user_answers`, `user_trainings`, `wk_media_view_sessions`, and badges are not treated as course-level without a verified course key.
- `user_activity_histories` is not used as full replacement for `wk_users_courses_actions`.
"""
(OUTPUT_DIR / "Hipotises_AGENTS.md").write_text(hyp_md, encoding="utf-8")
exported_files_AGENTS = pd.DataFrame([{"artifact": p.name, "path": str(p), "size_bytes": p.stat().st_size} for p in sorted(OUTPUT_DIR.glob("*_AGENTS.*"))])
display(exported_files_AGENTS)


## 12. Final summary

Built:
- structured EDA with table-by-table rough preprocessing;
- key restoration pipeline close to `Resolve_AGENT.ipynb`;
- expanded hypothesis checks H1-H8;
- interpretable feature blocks by source;
- final modeling table in grain `(user_id, course_id)`;
- audit tables for resolution, joins, missingness, leakage and outputs.

Unresolved handling:
- controlled by top-level `UNRESOLVED_MODE`;
- `keep` keeps unresolved rows and preserves unknown course-activity features as `NA`;
- `drop` removes unresolved rows from final export.
